<a href="https://colab.research.google.com/github/MinhNguyen19/honours_project/blob/main/code_submission/Untrained_data_llama_3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# import unsloth
from unsloth import FastLanguageModel
import pandas as pd
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, pipeline, BitsAndBytesConfig
import time
from google.colab import files
from datasets import Dataset
import gdown

In [ ]:
# Import data

url_1 = 'https://raw.githubusercontent.com/MinhNguyen19/honours_project/refs/heads/main/datasets_ft/preprocessed_datasets/politics/LIAR.feather'
url_2 = 'https://raw.githubusercontent.com/MinhNguyen19/honours_project/refs/heads/main/datasets_ft/preprocessed_datasets/politics/pheme.feather'

response = requests.get(url_1)
response.raise_for_status()
df_liar = pd.read_feather(BytesIO(response.content))

response = requests.get(url_2)
response.raise_for_status()
df_pheme = pd.read_feather(BytesIO(response.content))

df_liar_copy = df_liar.copy()
df_pheme_copy = df_pheme.copy()

In [ ]:
politics_id = '1tX1dMUS4R3qAqrZBreqvEFQH1UgdCGF1'
politics_url = f'https://drive.google.com/uc?id={politics_id}'
gdown.download(politics_url, "politics.feather", quiet=False)

# Get ISOT Politics and extract title column
df_politics = pd.read_feather("politics.feather")
df_politics_title = df_politics.copy()
df_politics_title = df_politics_title.drop(columns=['text'])
df_politics_title['text'] = df_politics_title['metadata'].apply(lambda x: x['metadata']['title'])
df_politics = df_politics_title.sample(frac=1, random_state=42)


In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Import model
model_8b, tokenizer_8b = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

In [ ]:
# Convert to Dataset
ds_pheme = Dataset.from_pandas(df_pheme)
ds_liar = Dataset.from_pandas(df_liar)
ds_politics = Dataset.from_pandas(df_politics)

In [ ]:
# Create prompts. Prompt engineering in here
def prompt_maker(text, examples=None):
    if examples is None:
        return_text = f'Determine whether the following news/claim is REAL or FAKE.\nNews: "{text}"\nAnswer with exactly one word: REAL or FAKE. \nAnswer: '
        return return_text
        # return f'Determine whether the following news/claim is REAL/FAKE.\nNews: "{text}"\nAnswer with exactly one word: REAL/FAKE. \nAnswer: '
    else:
        fewshot = ""
        num_examples = len(examples)
        for index, row in examples.iterrows():
            fewshot += f"\nNews: \"{row['text']}\"\nAnswer: {row['label']}\n"

        fewshot = fewshot.strip()
        return_text = (
            f"Determine whether the following news/claim is FAKE or REAL\n"
            f"{fewshot}\n"
            "Now determine. \n"
            f"News: \"{text}\"\n"
            "Answer with one word: FAKE or REAL\n"
            "Answer: "
        )

        return return_text



In [ ]:
# Get labels for fewshots

df_liar_true = df_liar[df_liar['label'] == 0]
df_liar_false = df_liar[df_liar['label'] == 1]

df_pheme_true = df_pheme[df_pheme['label'] == 0]
df_pheme_false = df_pheme[df_pheme['label'] == 1]

df_politics_true = df_politics[df_politics['label'] == 0]
df_politics_false = df_politics[df_politics['label'] == 1]

In [ ]:
import random

# Create examples for fewshots prompts
def combine_examples(df_true, df_false, num_examples):
    """
    Combines true and false examples for fewshots prompting

    Args:
        df_true (pd.DataFrame): DataFrame containing 'true' examples
        df_false (pd.DataFrame): DataFrame containing 'false' examples
        num_examples (int): The target number of examples to be sampled for each of the classes. 1 random example is included by default

    Returns:
        pd.DataFrame: A concatenated DataFrame of sampled true and false examples,
    """
    random_num = random.randint(0,1)
    # random_num = 0

    true_examples = df_true.sample(num_examples + random_num).replace({1: 'FAKE', 0: 'REAL'})
    false_examples = df_false.sample(num_examples + 1 - random_num).replace({1: 'FAKE', 0: 'REAL'})

    combined_examples = pd.concat([true_examples, false_examples])
    return combined_examples

In [ ]:
def predict_batch(model, tokenizer, batch, fewshots, df_true, df_false, num_examples, examples, batch_size, max_new_tokens, max_length=512):
    predictions = []
    raw_outputs = []

    for i in range(0, len(batch["text"]), batch_size):
        sub_texts = batch["text"][i:i+batch_size]
        if fewshots:
            if examples is None:
                examples = combine_examples(df_true, df_false, num_examples)

            prompts = [
                prompt_maker(text, examples)
                for text in sub_texts
            ]
        else:
            prompts = [
                prompt_maker(text)
                for text in sub_texts
            ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            padding="longest",
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                  **inputs,
                  max_new_tokens=max_new_tokens,
                  temperature=0.0,
                  do_sample=False,
                  top_k=1,
                  top_p=1,
                  repetition_penalty=1.1,
                  eos_token_id=tokenizer.eos_token_id
              )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        raw_outputs.extend(decoded)

        for out, prompt in zip(decoded, prompts):
            cleaned = out[len(prompt):].strip()
            text_upper = cleaned.upper()
            fake_count = text_upper.count("FAKE")
            real_count = text_upper.count("REAL")
            # fake_count = text_upper.count("0")
            # real_count = text_upper.count("1")
            if fake_count > real_count:
                predictions.append(1)
            elif real_count > fake_count:
                predictions.append(0)
            else:
                predictions.append(-1)

    return {"prediction": predictions, "raw_output": raw_outputs}


In [ ]:
def predict_result(ds, fewshots=False, df_true=None, df_false=None, num_examples=1, examples=None, batch_size=30, max_new_tokens=20):
    """
    Predicts the labels for a given dataset using the loaded language model. A wrapper for predict_batch.

    Args:
        ds (Dataset): The dataset containing the text for prediction.
        fewshots (bool, optional): Whether to use few-shot prompting. Defaults to False.
        df_true (pd.DataFrame, optional): DataFrame of true examples for few-shot prompting. Required if fewshots is True.
        df_false (pd.DataFrame, optional): DataFrame of false examples for few-shot prompting. Required if fewshots is True.
        num_examples (int, optional): Number of examples to use for few-shot prompting. Defaults to 1. Number of few-shot = num_examples x2 + 1. E.g. numexamples = 1 => 3 shots
        examples (pd.DataFrame, optional): Pre-combined examples for few-shot prompting. If provided, df_true, df_false, and num_examples are ignored.
        batch_size (int, optional): The number of samples to process in each batch. Defaults to 30.
        max_new_tokens (int, optional): The maximum number of new tokens to generate for each prediction. Defaults to 20.

    Returns:
        dict: A dictionary containing two lists:
              - 'prediction': List of predicted labels (0 for REAL, 1 for FAKE, -1 for undecided).
              - 'raw_output': List of raw model outputs.
    """
    results = {"prediction": [], "raw_output": []}

    start_time = time.time()

    for i in range(0, len(ds), batch_size):
        if i % 900 == 0:
            print(f"Reached {i+1} samples.")
        batch = ds[i:i+batch_size]
        batch_result = predict_batch(model_8b, tokenizer_8b, batch, fewshots, df_true, df_false, num_examples, examples, batch_size, max_new_tokens)
        results["prediction"].extend(batch_result["prediction"])
        results["raw_output"].extend(batch_result["raw_output"])

    end_time = time.time()
    elapsed_time = end_time - start_time

    print(f"Total inference time: {elapsed_time:.2f} seconds")
    print(f"Average time per row: {elapsed_time / len(ds):.2f} seconds")

    return results

In [ ]:
torch.cuda.empty_cache()

In [ ]:
ds_test = Dataset.from_pandas(df_pheme.head(60))

In [ ]:
# For zero-shot
results = predict_result(ds_test, max_new_tokens=8)#, fewshots=True, df_true=df_politics_true, df_false=df_politics_false, num_examples=1)

In [ ]:
# For fewshots
results = predict_result(ds_politics, max_new_tokens=8, fewshots=True, df_true=df_politics_true, df_false=df_politics_false, num_examples=1)

In [ ]:
results_df = pd.DataFrame(results)
results_df['prediction'].value_counts()

In [ ]:
print(results_df['raw_output'][0])

In [ ]:
undecided_predictions = results_df[results_df['prediction'] == -1].head(5)
for row in undecided_predictions.iterrows():
    print(row[1]['raw_output'])
    print('---')

In [ ]:
df_results = pd.DataFrame(results)

# Save to Colab runtime (temporary)
file_path = "/content/politics.csv"
df_results.to_csv(file_path, index=False)

from google.colab import files

files.download(file_path)


In [ ]:
from sklearn.metrics import classification_report

# Get the true labels from the original dataset
y_true = df_politics["label"]

# Get the predicted labels from the results dictionary
y_pred = results_df["prediction"]


y_true.reset_index(drop=True, inplace=True)

# Filter out the undecided predictions (-1) and corresponding true labels
mask = [p != -1 for p in y_pred]
y_true_filtered = [y_true[i] for i in range(len(y_true)) if mask[i]]
y_pred_filtered = [y_pred[i] for i in range(len(y_pred)) if mask[i]]


# Generate and print the classification report
print(classification_report(y_true_filtered, y_pred_filtered))